# 🧬 ClinVar — Missense Variant EDA & Dataset Pipeline
**TeknoFest 2026 | Sağlıkta Yapay Zeka**

Bu notebook:
1. `variant_summary.txt` dosyasını verimli şekilde yükler
2. Kapsamlı EDA yapar (sınıf dağılımı, eksik değer, varyant tipi, korelasyon)
3. Missense Pathogenic/Benign filtreli ML-hazır CSV oluşturur

---

## 0️⃣ Kurulum & Kütüphaneler

In [1]:
# Eksik kütüphaneleri yükle
!pip install missingno -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Plot stili
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
PALETTE = ['#2E86AB', '#E84855', '#3BB273', '#F4A261', '#9B5DE5']
print('✅ Kütüphaneler hazır')

✅ Kütüphaneler hazır


## 1️⃣ Veri Yükleme
> 700 MB dosyayı `usecols` ile sadece gerekli sütunları çekerek bellek tasarrufu yapıyoruz.

In [2]:
from google.colab import files
import io

import urllib.request
import gzip
import shutil
import os

FTP_URL   = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz'
GZ_PATH   = '/content/variant_summary.txt.gz'
FILE_PATH = '/content/variant_summary.txt'

def _progress(block_num, block_size, total_size):
    downloaded = block_num * block_size
    pct = min(downloaded / total_size * 100, 100) if total_size > 0 else 0
    bar = '█' * int(pct // 2) + '░' * (50 - int(pct // 2))
    print(f'\r  [{bar}] {pct:5.1f}%  ({downloaded/1e6:.0f} / {total_size/1e6:.0f} MB)', end='')

if os.path.exists(FILE_PATH):
    print(f'✅ {FILE_PATH} zaten mevcut, indirme atlanıyor.')
else:
    if not os.path.exists(GZ_PATH):
        print('📥 ClinVar FTP\'den indiriliyor...')
        print(f'   {FTP_URL}')
        urllib.request.urlretrieve(FTP_URL, GZ_PATH, reporthook=_progress)
        print(f'\n✅ İndirme tamamlandı → {GZ_PATH}')
    else:
        print('✅ .gz zaten mevcut, tekrar indirilmeyecek.')

    print('📦 Sıkıştırma açılıyor (.gz → .txt) ...')
    with gzip.open(GZ_PATH, 'rb') as f_in, open(FILE_PATH, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f'✅ Açıldı → {FILE_PATH}')
    print(f'   Dosya boyutu: {os.path.getsize(FILE_PATH)/1e6:.0f} MB')

print(f'\n📂 Kullanılacak dosya: {FILE_PATH}')

📥 ClinVar FTP'den indiriliyor...
   https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
  [██████████████████████████████████████████████████] 100.0%  (435 / 435 MB)
✅ İndirme tamamlandı → /content/variant_summary.txt.gz
📦 Sıkıştırma açılıyor (.gz → .txt) ...
✅ Açıldı → /content/variant_summary.txt
   Dosya boyutu: 3897 MB

📂 Kullanılacak dosya: /content/variant_summary.txt


In [3]:
# İlk satırı göster — sütun isimlerini kontrol et
with open(FILE_PATH, 'r', encoding='utf-8', errors='replace') as f:
    header = f.readline()

cols_all = header.strip().split('\t')
print(f'Toplam sütun sayısı: {len(cols_all)}')
print()
for i, c in enumerate(cols_all):
    print(f'  [{i:2d}] {c}')

Toplam sütun sayısı: 43

  [ 0] #AlleleID
  [ 1] Type
  [ 2] Name
  [ 3] GeneID
  [ 4] GeneSymbol
  [ 5] HGNC_ID
  [ 6] ClinicalSignificance
  [ 7] ClinSigSimple
  [ 8] LastEvaluated
  [ 9] RS# (dbSNP)
  [10] nsv/esv (dbVar)
  [11] RCVaccession
  [12] PhenotypeIDS
  [13] PhenotypeList
  [14] Origin
  [15] OriginSimple
  [16] Assembly
  [17] ChromosomeAccession
  [18] Chromosome
  [19] Start
  [20] Stop
  [21] ReferenceAllele
  [22] AlternateAllele
  [23] Cytogenetic
  [24] ReviewStatus
  [25] NumberSubmitters
  [26] Guidelines
  [27] TestedInGTR
  [28] OtherIDs
  [29] SubmitterCategories
  [30] VariationID
  [31] PositionVCF
  [32] ReferenceAlleleVCF
  [33] AlternateAlleleVCF
  [34] SomaticClinicalImpact
  [35] SomaticClinicalImpactLastEvaluated
  [36] ReviewStatusClinicalImpact
  [37] Oncogenicity
  [38] OncogenicityLastEvaluated
  [39] ReviewStatusOncogenicity
  [40] SCVsForAggregateGermlineClassification
  [41] SCVsForAggregateSomaticClinicalImpact
  [42] SCVsForAggregateOncogenicit

In [ ]:
COLS_NEEDED = [
    'Type',
    'ClinicalSignificance',
    'ReviewStatus',
    'GeneSymbol',
    'Chromosome',
    'Start',
    'Stop',
    'Assembly',
    'OriginSimple',
    'PhenotypeList',
    'Cytogenetic',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
    'LastEvaluated',
    # bilgi sütunları
    '#AlleleID',
    'Name',
    'GeneID',
    'RCVaccession',
    'VariationID',
    'RS# (dbSNP)',
    'PhenotypeIDS',
    'NumberSubmitters',
]

COLS_LOAD = [c for c in COLS_NEEDED if c in cols_all]
missing_cols = set(COLS_NEEDED) - set(COLS_LOAD)
if missing_cols:
    print(f'⚠️  Bulunamayan sütunlar: {missing_cols}')

print(f'📥 Yükleniyor... ({len(COLS_LOAD)} sütun, lütfen bekleyin)')
df_raw = pd.read_csv(
    FILE_PATH,
    sep='\t',
    usecols=COLS_LOAD,
    low_memory=False,
    encoding='utf-8',
    on_bad_lines='skip'
)
print(f'✅ Yüklendi! Şekil: {df_raw.shape}')
print(f'   Bellek kullanımı: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_raw.head(3)

📥 Yükleniyor... (22 sütun, lütfen bekleyin)


---
## 2️⃣ EDA — Genel Bakış

In [ ]:
print('=' * 55)
print('📊 GENEL İSTATİSTİKLER')
print('=' * 55)
print(f'Toplam varyant sayısı : {len(df_raw):,}')
print(f'Sütun sayısı          : {df_raw.shape[1]}')
print()

if 'Type' in df_raw.columns:
    print('--- Varyant Tipleri (ilk 10) ---')
    print(df_raw['Type'].value_counts().head(10).to_string())
    print()

if 'ClinicalSignificance' in df_raw.columns:
    print('--- ClinicalSignificance (ilk 15) ---')
    print(df_raw['ClinicalSignificance'].value_counts().head(15).to_string())

### 2.1 🔬 Varyant Tipi Dağılımı

In [ ]:
if 'Type' not in df_raw.columns:
    print('Type sütunu bulunamadı, atlanıyor.')
else:
    type_counts = df_raw['Type'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    ax = axes[0]
    bars = ax.barh(type_counts.index[:12], type_counts.values[:12],
                   color=PALETTE[0], edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Varyant Sayısı')
    ax.set_title('Varyant Tipi Dağılımı (ilk 12)', fontweight='bold', pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for bar in bars:
        w = bar.get_width()
        ax.text(w * 1.01, bar.get_y() + bar.get_height()/2,
                f'{w:,.0f}', va='center', fontsize=8)

    # Pie — sadece ilk 6 tipi göster, kalanı 'Diğer'
    ax2 = axes[1]
    top6 = type_counts.head(6)
    other = type_counts.iloc[6:].sum()
    pie_data = pd.concat([top6, pd.Series({'Diğer': other})])
    ax2.pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
            colors=PALETTE + ['#aaa'], startangle=140,
            textprops={'fontsize': 8})
    ax2.set_title('Varyant Tipi Oranları', fontweight='bold', pad=12)

    plt.tight_layout()
    plt.savefig('eda_variant_type.png', bbox_inches='tight')
    plt.show()
    print(f'\n💾 eda_variant_type.png kaydedildi')

### 2.2 🎯 Sınıf Dağılımı (Pathogenic / Benign)

In [ ]:
if 'ClinicalSignificance' not in df_raw.columns:
    print('ClinicalSignificance sütunu bulunamadı, atlanıyor.')
else:
    # Tüm veri setindeki sınıf dağılımı
    sig_counts = df_raw['ClinicalSignificance'].value_counts()

    # Renk haritası — önemli kategorileri vurgula
    color_map = {
        'Pathogenic': '#E84855',
        'Likely pathogenic': '#F4845F',
        'Benign': '#2E86AB',
        'Likely benign': '#5FBEDF',
        'Uncertain significance': '#9B9B9B',
    }
    colors = [color_map.get(k, '#cccccc') for k in sig_counts.index[:15]]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Tüm kategoriler
    ax = axes[0]
    ax.barh(sig_counts.index[:15][::-1], sig_counts.values[:15][::-1],
            color=colors[::-1], edgecolor='white')
    ax.set_xlabel('Varyant Sayısı')
    ax.set_title('ClinVar Klinik Anlamlılık Dağılımı (Tüm Varyantlar)',
                 fontweight='bold', pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    # Sadece Pathogenic / Benign grubu
    ax2 = axes[1]
    target_map = {
        'Pathogenic': 'Pathogenic\n(P)',
        'Likely pathogenic': 'Likely\nPathogenic (LP)',
        'Benign': 'Benign\n(B)',
        'Likely benign': 'Likely\nBenign (LB)',
        'Uncertain significance': 'VUS'
    }
    target_counts = {
        label: sig_counts.get(k, 0)
        for k, label in target_map.items()
    }
    tc = pd.Series(target_counts)
    bars2 = ax2.bar(tc.index, tc.values,
                    color=['#E84855','#F4845F','#2E86AB','#5FBEDF','#9B9B9B'],
                    edgecolor='white', linewidth=0.5)
    for bar in bars2:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h * 1.01,
                 f'{h:,.0f}', ha='center', fontsize=9, fontweight='bold')
    ax2.set_ylabel('Varyant Sayısı')
    ax2.set_title('P / LP / B / LB / VUS Karşılaştırması', fontweight='bold', pad=12)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    plt.tight_layout()
    plt.savefig('eda_class_distribution.png', bbox_inches='tight')
    plt.show()

    # Özet tablo
    print('\n📋 Hedef Sınıf Özeti:')
    p  = sig_counts.get('Pathogenic', 0)
    lp = sig_counts.get('Likely pathogenic', 0)
    b  = sig_counts.get('Benign', 0)
    lb = sig_counts.get('Likely benign', 0)
    vus = sig_counts.get('Uncertain significance', 0)
    total = p + lp + b + lb
    print(f'  Pathogenic (P)         : {p:>8,}')
    print(f'  Likely Pathogenic (LP) : {lp:>8,}')
    print(f'  P + LP (pozitif sınıf) : {p+lp:>8,}  ({(p+lp)/max(total,1)*100:.1f}%)')
    print(f'  Benign (B)             : {b:>8,}')
    print(f'  Likely Benign (LB)     : {lb:>8,}')
    print(f'  B + LB (negatif sınıf) : {b+lb:>8,}  ({(b+lb)/max(total,1)*100:.1f}%)')
    print(f'  Uncertain Sig. (VUS)   : {vus:>8,}  (ML\'ye dahil edilmeyecek)')

### 2.3 ❓ Eksik Değer Analizi

In [ ]:
# Eksik değerleri '-' ve 'na' gibi placeholder'larla beraber tespit et
MISSING_PLACEHOLDERS = ['-', 'na', 'n/a', 'NA', 'N/A', 'none', 'None', '.']

df_check = df_raw.replace(MISSING_PLACEHOLDERS, np.nan)

missing = df_check.isnull().sum()
missing_pct = (missing / len(df_check) * 100).round(2)
missing_df = pd.DataFrame({
    'Eksik Sayı': missing,
    'Eksik (%)': missing_pct,
    'Dolu Sayı': len(df_check) - missing,
    'Veri Tipi': df_check.dtypes
}).sort_values('Eksik (%)', ascending=False)

print('📋 Eksik Değer Özeti:')
display(missing_df)

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — eksik oran
ax = axes[0]
colors_miss = ['#E84855' if p > 50 else '#F4A261' if p > 20 else '#3BB273'
               for p in missing_pct.values]
ax.barh(missing_pct.index, missing_pct.values, color=colors_miss)
ax.axvline(20, color='orange', linestyle='--', linewidth=1, label='%20 eşiği')
ax.axvline(50, color='red', linestyle='--', linewidth=1, label='%50 eşiği')
ax.set_xlabel('Eksik Değer Oranı (%)')
ax.set_title('Sütun Bazında Eksik Değer Oranları', fontweight='bold')
ax.legend(fontsize=8)
for i, (col, pct) in enumerate(missing_pct.items()):
    if pct > 0:
        ax.text(pct + 0.5, i, f'{pct:.1f}%', va='center', fontsize=8)

# Missingno matris
axes[1].set_visible(False)  # missingno kendi figure oluşturur
plt.tight_layout()
plt.savefig('eda_missing_bar.png', bbox_inches='tight')
plt.show()

# Missingno matrix
fig2 = plt.figure(figsize=(10, 5))
msno.matrix(df_check, sparkline=False, figsize=(10, 5), fontsize=9,
            color=(0.18, 0.53, 0.67))
plt.title('Eksik Değer Matrisi (beyaz = eksik)', fontweight='bold')
plt.savefig('eda_missing_matrix.png', bbox_inches='tight')
plt.show()
print('💾 Görseller kaydedildi.')

### 2.4 🔗 Özellik Korelasyonları

In [ ]:
# Sayısal sütunlar için korelasyon — önce missense filtresini uygula
# Geçici label ekle
df_corr = df_raw.copy()
df_corr = df_corr.replace(MISSING_PLACEHOLDERS, np.nan)

if 'ClinicalSignificance' in df_corr.columns:
    label_map = {
        'Pathogenic': 1, 'Likely pathogenic': 1,
        'Benign': 0, 'Likely benign': 0
    }
    df_corr['label'] = df_corr['ClinicalSignificance'].map(label_map)

# Sayısal sütunları seç
num_cols = df_corr.select_dtypes(include=[np.number]).columns.tolist()
print(f'Sayısal sütunlar: {num_cols}')

if len(num_cols) >= 2:
    corr_matrix = df_corr[num_cols].corr()

    fig, ax = plt.subplots(figsize=(max(6, len(num_cols)), max(5, len(num_cols))))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, mask=mask, annot=True, fmt='.2f',
        cmap='coolwarm', center=0, square=True,
        linewidths=0.5, cbar_kws={'shrink': 0.8},
        ax=ax, annot_kws={'size': 9}
    )
    ax.set_title('Sayısal Özellikler Arası Korelasyon Isı Haritası', fontweight='bold', pad=14)
    plt.tight_layout()
    plt.savefig('eda_correlation.png', bbox_inches='tight')
    plt.show()

    if 'label' in num_cols:
        print('\n📊 Label ile korelasyonlar:')
        label_corr = corr_matrix['label'].drop('label').sort_values(key=abs, ascending=False)
        print(label_corr.to_string())
else:
    print('⚠️  Yeterli sayısal sütun yok. Korelasyon atlanıyor.')
    print('   (variant_summary.txt ağırlıklı kategorik — asıl korelasyon filtreli CSV sonrası yapılacak)')

### 2.5 📈 Ek EDA — Review Status & Gen Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ReviewStatus dağılımı
if 'ReviewStatus' in df_raw.columns:
    rs = df_raw['ReviewStatus'].value_counts()
    axes[0].barh(rs.index[:10][::-1], rs.values[:10][::-1], color=PALETTE[2])
    axes[0].set_xlabel('Varyant Sayısı')
    axes[0].set_title('Review Status Dağılımı (Kanıt Kalitesi)', fontweight='bold')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
else:
    axes[0].text(0.5, 0.5, 'ReviewStatus\nsütunu yok', ha='center', va='center')

# En çok varyantlı genler
if 'GeneSymbol' in df_raw.columns:
    gene_counts = df_raw['GeneSymbol'].value_counts().head(20)
    axes[1].barh(gene_counts.index[::-1], gene_counts.values[::-1], color=PALETTE[3])
    axes[1].set_xlabel('Varyant Sayısı')
    axes[1].set_title('En Çok Varyantlı 20 Gen', fontweight='bold')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
else:
    axes[1].text(0.5, 0.5, 'GeneSymbol\nsütunu yok', ha='center', va='center')

plt.tight_layout()
plt.savefig('eda_review_gene.png', bbox_inches='tight')
plt.show()

In [ ]:
# Pathogenic ve Benign varyantların review status karşılaştırması
if 'ReviewStatus' in df_raw.columns and 'ClinicalSignificance' in df_raw.columns:
    df_pb = df_raw[df_raw['ClinicalSignificance'].isin(
        ['Pathogenic', 'Likely pathogenic', 'Benign', 'Likely benign']
    )].copy()
    df_pb['Group'] = df_pb['ClinicalSignificance'].map({
        'Pathogenic': 'Pathogenic/LP',
        'Likely pathogenic': 'Pathogenic/LP',
        'Benign': 'Benign/LB',
        'Likely benign': 'Benign/LB'
    })

    pivot = df_pb.groupby(['ReviewStatus', 'Group']).size().unstack(fill_value=0)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index[:8]]

    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='barh', ax=ax, color=['#E84855', '#2E86AB'], edgecolor='white')
    ax.set_xlabel('Varyant Sayısı')
    ax.set_title('Review Status × Sınıf Dağılımı', fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(title='Sınıf')
    plt.tight_layout()
    plt.savefig('eda_review_vs_class.png', bbox_inches='tight')
    plt.show()

---
## 3️⃣ Missense Filtresi & Etiketleme
> Şimdi modeli eğiteceğimiz temiz veri setini oluşturuyoruz.

In [ ]:
# ─── ADIM 1: Varyant tipi filtresi ─────────────────────────────────────────
# ClinVar'da missense varyantlar 'single nucleotide variant' olarak geçer
# Bunu daraltmak için klinik anlamlılıkla birlikte filtrele

if 'Type' in df_raw.columns:
    df_snv = df_raw[df_raw['Type'] == 'single nucleotide variant'].copy()
    print(f'SNV (single nucleotide variant) : {len(df_snv):,}')
else:
    df_snv = df_raw.copy()
    print('Type sütunu yok, tüm varyantlar alınıyor.')

# ─── ADIM 2: Klinik anlamlılık filtresi ─────────────────────────────────────
TARGET_CLASSES = ['Pathogenic', 'Likely pathogenic', 'Benign', 'Likely benign']

df_filtered = df_snv[df_snv['ClinicalSignificance'].isin(TARGET_CLASSES)].copy()
print(f'Pathogenic + Benign grubu       : {len(df_filtered):,}')

# ─── ADIM 3: Etiket oluştur ──────────────────────────────────────────────────
label_map = {
    'Pathogenic': 1,
    'Likely pathogenic': 1,
    'Benign': 0,
    'Likely benign': 0
}
df_filtered['label'] = df_filtered['ClinicalSignificance'].map(label_map)

print(f'\nEtiket dağılımı:')
print(df_filtered['label'].value_counts().rename({0: 'Benign (0)', 1: 'Pathogenic (1)'}).to_string())

ratio = df_filtered['label'].value_counts()
pos_pct = ratio.get(1, 0) / len(df_filtered) * 100
print(f'\n  ➤ Pozitif (Pathogenic) oranı: {pos_pct:.1f}%')
print(f'  ➤ Negatif (Benign)    oranı: {100-pos_pct:.1f}%')

In [ ]:
# ─── ADIM 4: Review status kalite filtresi (opsiyonel ama önerilir) ─────────
# Yüksek kanıt kaliteli varyantları tercih etmek için

HIGH_QUALITY_REVIEWS = [
    'reviewed by expert panel',
    'practice guideline',
    'criteria provided, multiple submitters, no conflicts',
    'criteria provided, single submitter',
]

if 'ReviewStatus' in df_filtered.columns:
    rs_counts = df_filtered['ReviewStatus'].value_counts()
    print('Review status dağılımı (filtrelenmiş veri):')
    print(rs_counts.to_string())

    print('\n⚙️  Kalite filtresi uygulanıyor...')
    df_hq = df_filtered[df_filtered['ReviewStatus'].isin(HIGH_QUALITY_REVIEWS)].copy()
    df_all_q = df_filtered.copy()  # Filtre uygulanmamış versiyon

    print(f'  Yüksek kalite (expert/multi-submitter) : {len(df_hq):,}')
    print(f'  Tüm review statüsleri                  : {len(df_all_q):,}')
    print('\n  👉 İkisini de kaydediyoruz.')
else:
    df_hq = df_filtered.copy()
    df_all_q = df_filtered.copy()

---
## 4️⃣ Filtrelenmiş EDA (Missense P/B Veri Seti)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) Sınıf dengesi
ax = axes[0]
vals = df_filtered['label'].value_counts().sort_index()
bars = ax.bar(['Benign (0)', 'Pathogenic (1)'], vals.values,
              color=['#2E86AB', '#E84855'], edgecolor='white', linewidth=0.5)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h * 1.01,
            f'{h:,}', ha='center', fontweight='bold')
ax.set_title('Sınıf Dengesi\n(Missense P/B)', fontweight='bold')
ax.set_ylabel('Varyant Sayısı')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# (2) Kalite bazında dağılım
ax2 = axes[1]
if 'ReviewStatus' in df_filtered.columns:
    rs_label = df_filtered.groupby(['ReviewStatus', 'label']).size().unstack(fill_value=0)
    rs_label = rs_label.loc[rs_label.sum(axis=1).nlargest(6).index]
    rs_label.plot(kind='barh', ax=ax2, color=['#2E86AB','#E84855'], edgecolor='white')
    ax2.set_title('Review Status × Sınıf\n(Missense P/B)', fontweight='bold')
    ax2.legend(['Benign', 'Pathogenic'], fontsize=8)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
else:
    ax2.text(0.5, 0.5, 'ReviewStatus yok', ha='center', va='center')

# (3) Assembly dağılımı
ax3 = axes[2]
if 'Assembly' in df_filtered.columns:
    asm = df_filtered['Assembly'].value_counts()
    ax3.pie(asm.values, labels=asm.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=90)
    ax3.set_title('Genome Assembly\nDağılımı', fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'Assembly yok', ha='center', va='center')

plt.tight_layout()
plt.savefig('eda_filtered_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# En çok P/B varyantı olan genler
if 'GeneSymbol' in df_filtered.columns:
    gene_pb = df_filtered.groupby(['GeneSymbol', 'label']).size().unstack(fill_value=0)
    gene_pb.columns = ['Benign', 'Pathogenic']
    gene_pb['Toplam'] = gene_pb.sum(axis=1)
    top_genes = gene_pb.nlargest(20, 'Toplam')

    fig, ax = plt.subplots(figsize=(12, 8))
    top_genes[['Benign','Pathogenic']].plot(
        kind='barh', ax=ax,
        color=['#2E86AB','#E84855'], edgecolor='white', linewidth=0.5
    )
    ax.set_xlabel('Varyant Sayısı')
    ax.set_title('Missense P/B Varyantları — En Çok 20 Gen', fontweight='bold', pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(['Benign', 'Pathogenic'])
    plt.tight_layout()
    plt.savefig('eda_gene_pb.png', bbox_inches='tight')
    plt.show()

---
## 5️⃣ CSV Oluşturma

In [ ]:
# Her sınıftan 1000'er örnek → toplam 2000 dengeli veri seti

# Önce en kaliteli olanları önceliklendir
QUALITY_ORDER = [
    'practice guideline',
    'reviewed by expert panel',
    'criteria provided, multiple submitters, no conflicts',
]

df_final = df_raw[
    (df_raw['Type'] == 'single nucleotide variant') &
    (df_raw['ClinicalSignificance'].isin(TARGET_CLASSES)) &
    (df_raw['Assembly'] == 'GRCh38')
].copy()

df_final['label'] = df_final['ClinicalSignificance'].map({
    'Pathogenic': 1, 'Likely pathogenic': 1,
    'Benign': 0,     'Likely benign': 0
})

# Kalite sırasına göre sırala — en iyi yıldızlar üstte
df_final['quality_rank'] = df_final['ReviewStatus'].map({
    'practice guideline': 0,
    'reviewed by expert panel': 1,
    'criteria provided, multiple submitters, no conflicts': 2,
    'criteria provided, single submitter': 3,
    'no assertion criteria provided': 4,
}).fillna(5)

df_final = df_final.sort_values('quality_rank')

# Her sınıftan en kaliteli 1000'i al
N_PER_CLASS = 1000

df_path = df_final[df_final['label'] == 1].head(N_PER_CLASS)
df_ben  = df_final[df_final['label'] == 0].head(N_PER_CLASS)

df_2k = pd.concat([df_path, df_ben]).sample(frac=1, random_state=42).reset_index(drop=True)
df_2k = df_2k.drop(columns=['quality_rank'])

print(f'✅ Final veri seti: {len(df_2k):,} satır')
print(df_2k['label'].value_counts().rename({0:'Benign', 1:'Pathogenic'}))
print(f"\nKalite dağılımı:")
print(df_2k['ReviewStatus'].value_counts())

df_2k.to_csv('clinvar_missense_2k.csv', index=False)
print('\n💾 clinvar_missense_2k.csv kaydedildi!')

In [ ]:
ML_COLS = [            # modele gidecek
    'GeneSymbol',
    'Cytogenetic',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
    'OriginSimple',
    'PhenotypeList',
    'LastEvaluated',
]

INFO_COLS = [          # CSV'de bilgi için, modele gitmiyor
    '#AlleleID',
    'Name',
    'GeneID',
    'RCVaccession',
    'VariationID',
    'RS# (dbSNP)',
    'PhenotypeIDS',
    'Chromosome',
    'Start',
    'Stop',
    'Assembly',
    'ReviewStatus',
    'NumberSubmitters',
    'ClinicalSignificance',
]

TARGET_COL = ['label']

ML_COLS   = [c for c in ML_COLS   if c in df_2k.columns]
INFO_COLS = [c for c in INFO_COLS if c in df_2k.columns]
ALL_COLS  = ML_COLS + INFO_COLS + TARGET_COL

df_ml = df_2k[ALL_COLS].copy()
df_ml = df_ml.replace(MISSING_PLACEHOLDERS, np.nan)

print(f'✅ ML veri seti hazır!')
print(f'   Satır sayısı        : {len(df_ml):,}')
print(f'   ML özellikleri ({len(ML_COLS)}) : {ML_COLS}')
print(f'   Bilgi sütunları ({len(INFO_COLS)}): {INFO_COLS}')
print(f'   Hedef               : label (0=Benign, 1=Pathogenic)')
display(df_ml.head())

df_ml.to_csv('clinvar_missense_2k.csv', index=False)
print('💾 clinvar_missense_2k.csv kaydedildi!'

In [ ]:
# ─── CSV kaydetme ────────────────────────────────────────────────────────────

# 1. Tüm review status — geniş veri seti
OUT_ALL = 'clinvar_missense_all.csv'
df_ml.to_csv(OUT_ALL, index=False)
print(f'💾 {OUT_ALL:40s} → {len(df_ml):>8,} satır')

# 2. Yüksek kalite — sadece expert/multi-submitter
if 'ReviewStatus' in df_hq.columns:
    df_hq_ml = df_hq[[c for c in ALL_COLS if c in df_hq.columns]].replace(MISSING_PLACEHOLDERS, np.nan)
    OUT_HQ = 'clinvar_missense_highquality.csv'
    df_hq_ml.to_csv(OUT_HQ, index=False)
    print(f'💾 {OUT_HQ:40s} → {len(df_hq_ml):>8,} satır')

# 3. Yüksek kalite + GRCh38 (en temiz)
if 'Assembly' in df_hq_ml.columns:
    df_grch38 = df_hq_ml[df_hq_ml['Assembly'] == 'GRCh38'].copy()
    OUT_38 = 'clinvar_missense_hq_grch38.csv'
    df_grch38.to_csv(OUT_38, index=False)
    print(f'💾 {OUT_38:40s} → {len(df_grch38):>8,} satır')

print('\n✅ Tüm CSV dosyaları oluşturuldu!')

In [ ]:
# ─── Google Drive'a kaydet (opsiyonel) ───────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(OUT_ALL, '/content/drive/MyDrive/clinvar_missense_all.csv')
# shutil.copy(OUT_HQ,  '/content/drive/MyDrive/clinvar_missense_highquality.csv')
# print('Drive\'a kopyalandı!')

# ─── Ya da doğrudan indir ────────────────────────────────────────────────────
# from google.colab import files
# files.download(OUT_ALL)
# files.download(OUT_HQ)

---
## 6️⃣ Özet Rapor

In [ ]:
print('=' * 60)
print('📊 EDA & DATASET PIPELINE — ÖZET RAPOR')
print('=' * 60)

print(f'\n🔵 Ham Veri')
print(f'   Toplam varyant            : {len(df_raw):>10,}')

if 'Type' in df_raw.columns:
    n_snv = (df_raw['Type'] == 'single nucleotide variant').sum()
    print(f'   SNV (missense adayı)      : {n_snv:>10,}  ({n_snv/len(df_raw)*100:.1f}%)')

print(f'\n🟢 Filtrelenmiş (P+LP / B+LB)')
print(f'   Toplam                    : {len(df_filtered):>10,}')

n_path = (df_filtered['label'] == 1).sum()
n_ben  = (df_filtered['label'] == 0).sum()
print(f'   Pathogenic/LP (1)         : {n_path:>10,}  ({n_path/len(df_filtered)*100:.1f}%)')
print(f'   Benign/LB     (0)         : {n_ben:>10,}  ({n_ben/len(df_filtered)*100:.1f}%)')

if 'ReviewStatus' in df_filtered.columns:
    print(f'\n🟡 Yüksek Kalite (expert/multi-submitter)')
    print(f'   Toplam                    : {len(df_hq):>10,}')

print(f'\n📁 Oluşturulan CSV Dosyaları')
print(f'   clinvar_missense_all.csv          → tüm review statüsleri')
print(f'   clinvar_missense_highquality.csv  → expert/multi-submitter')
if 'Assembly' in df_filtered.columns:
    print(f'   clinvar_missense_hq_grch38.csv    → yüksek kalite + GRCh38')

print(f'\n📊 Oluşturulan EDA Görselleri')
for g in ['eda_variant_type.png','eda_class_distribution.png',
           'eda_missing_bar.png','eda_missing_matrix.png',
           'eda_review_gene.png','eda_review_vs_class.png',
           'eda_filtered_overview.png','eda_gene_pb.png','eda_correlation.png']:
    print(f'   {g}')

print('\n' + '=' * 60)
print('✅ Pipeline tamamlandı. Sıradaki adım: Model eğitimi (model_training.ipynb)')
print('=' * 60)